##1. Environment Setup and Library Installation
Install the Python libraries required for data ingestion and processing

In [1]:
!pip install pandas openpyxl yfinance

##2. RBA Data Ingestion - Automated File Retrieval
Retrieve and preserve the RBA F1 Excel workbook using Python.


In [2]:

import requests
import os
from datetime import datetime
from zoneinfo import ZoneInfo

url = "https://www.rba.gov.au/statistics/tables/xls/f01d.xlsx"

os.makedirs("raw", exist_ok=True)

timestamp = datetime.now(
    ZoneInfo("Australia/Sydney")
).strftime("%Y%m%d_%H%M%S")

filename = f"raw/rba_F1_{timestamp}.xlsx"

response = requests.get(url, timeout=30)
response.raise_for_status()

if response.content[:2] != b"PK":
    raise ValueError("Downloaded file is not a valid Excel workbook")

with open(filename, "wb") as file:
    file.write(response.content)

print("RBA file downloaded successfully!")
print("Saved as:", filename)


RBA file downloaded successfully!
Saved as: raw/rba_F1_20261014_222754.xlsx


##3. RBA Data Extraction and Date Filtering
Extract the daily cash rate and select the 120-calendar-day analysis period.

In [4]:

import pandas as pd

rba_latest = pd.read_excel(
    filename,
    usecols="A,D",
    skiprows=11,
    header=None,
    names=["Date", "Cash_Rate_Percent"]
)

rba_latest["Date"] = pd.to_datetime(
    rba_latest["Date"],
    errors="coerce"
)

rba_latest_120 = rba_latest[
    rba_latest["Date"].between(
        "2026-06-16", "2026-10-13"
    )
]

print(rba_latest_120.tail())
print("Total rows:", len(rba_latest_120))
print("Missing cash rates:",
      rba_latest_120["Cash_Rate_Percent"].isna().sum())


           Date  Cash_Rate_Percent
3989 2026-10-07                4.6
3990 2026-10-08                4.6
3991 2026-10-09                4.6
3992 2026-10-12                4.6
3993 2026-10-13                4.6
Total rows: 86
Missing cash rates: 0


##4. S&P/ASX 200 Market Data Ingestion
Retrieve historical ASX 200 data from Yahoo Finance and save it as a CSV file


In [5]:

import yfinance as yf
import os

os.makedirs("raw", exist_ok=True)

asx200 = yf.Ticker("^AXJO").history(
    start="2026-06-16",
    end="2026-10-14",
    interval="1d",
    auto_adjust=False
)

if asx200.empty:
    print("No ASX 200 data was retrieved.")
else:
    asx200.to_csv("raw/asx200_retrieved.csv")

    print(asx200[["Close"]].head())
    print("Total observations:", len(asx200))
    print("Latest date:", asx200.index.max())


                                 Close
Date                                  
2026-06-16 00:00:00+10:00  8917.700195
2026-06-17 00:00:00+10:00  8966.299805
2026-06-18 00:00:00+10:00  8911.099609
2026-06-19 00:00:00+10:00  8828.700195
2026-06-22 00:00:00+10:00  8816.099609
Total observations: 86
Latest date: 2026-10-13 00:00:00+11:00


##5. Initial Data Quality Assessment
Check both datasets for missing observations and closing prices


In [6]:

print("RBA DATA QUALITY CHECK")
print("Total observations:", len(rba_latest_120))
print(
    "Missing cash rates:",
    rba_latest_120["Cash_Rate_Percent"].isna().sum()
)

print()

print("ASX 200 DATA QUALITY CHECK")
print("Total observations:", len(asx200))
print(
    "Missing closing prices:",
    asx200["Close"].isna().sum()
)

print("First date:", asx200.index.min())
print("Latest date:", asx200.index.max())


RBA DATA QUALITY CHECK
Total observations: 86
Missing cash rates: 0

ASX 200 DATA QUALITY CHECK
Total observations: 86
Missing closing prices: 0
First date: 2026-06-16 00:00:00+10:00
Latest date: 2026-10-13 00:00:00+11:00


##6. Missing-Value Detection and Logging
Identify and record missing RBA cash-rate observations.

In [7]:

import os

os.makedirs("logs", exist_ok=True)

missing_rba_latest = rba_latest_120[
    rba_latest_120["Cash_Rate_Percent"].isna()
].copy()

missing_rba_latest["Issue"] = "Missing cash rate"

missing_rba_latest.to_csv(
    "logs/rba_missing_values_latest.csv",
    index=False
)

print(missing_rba_latest)
print("Missing records logged:", len(missing_rba_latest))


Empty DataFrame
Columns: [Date, Cash_Rate_Percent, Issue]
Index: []
Missing records logged: 0


##7. Simulated Download Failure and Error Logging
Test the pipeline's response to an invalid URL and record the resulting HTTP error

In [8]:

import requests
import csv
import os
from datetime import datetime
from zoneinfo import ZoneInfo

test_url = "https://www.rba.gov.au/invalid-test-file.xlsx"

try:
    response = requests.get(test_url, timeout=15)
    response.raise_for_status()

    print("Test connection successful.")

except requests.RequestException as error:
    os.makedirs("logs", exist_ok=True)

    log_file = "logs/rba_download_errors.csv"

    with open(log_file, "a", newline="") as file:
        writer = csv.writer(file)

        if file.tell() == 0:
            writer.writerow(["Time", "Source", "Error"])

        writer.writerow([
            datetime.now(
                ZoneInfo("Australia/Sydney")
            ).isoformat(),
            test_url,
            str(error)
        ])

    print("Simulated download failure logged.")
    print(error)

print("Error log exists:",
      os.path.exists("logs/rba_download_errors.csv"))


Simulated download failure logged.
404 Client Error: Not Found for url: https://www.rba.gov.au/invalid-test-file.xlsx
Error log exists: True


##8. Proof-of-Concept Execution Summary
  Display the final data-quality results and generated files.

In [9]:

print("BUSA3022 - PART 1 PROOF OF CONCEPT")
print("----------------------------------")

print("RBA observations:", len(rba_latest_120))
print(
    "Missing RBA values:",
    rba_latest_120["Cash_Rate_Percent"].isna().sum()
)

print("ASX 200 observations:", len(asx200))
print(
    "Missing ASX closing prices:",
    asx200["Close"].isna().sum()
)

print("Missing RBA records logged:", len(missing_rba_latest))

print("\nFiles created:")

for folder in ["raw", "logs"]:
    print(folder + "/")

    for name in sorted(os.listdir(folder)):
        print(" -", name)

print("\nDownload failure log:")

error_log = pd.read_csv("logs/rba_download_errors.csv")
print(error_log.tail(1).to_string(index=False))


BUSA3022 - PART 1 PROOF OF CONCEPT
----------------------------------
RBA observations: 86
Missing RBA values: 0
ASX 200 observations: 86
Missing ASX closing prices: 0
Missing RBA records logged: 0

Files created:
raw/
 - asx200_retrieved.csv
 - rba_F1_20261014_222754.xlsx
 - rba_F1_20261014_222817.xlsx
logs/
 - rba_download_errors.csv
 - rba_missing_values_latest.csv

Download failure log:
                            Time                                        Source                                                                              Error
2026-10-14T22:30:28.324737+11:00 https://www.rba.gov.au/invalid-test-file.xlsx 404 Client Error: Not Found for url: https://www.rba.gov.au/invalid-test-file.xlsx
